# Native AHR v5 — shared definitions

Purpose: import frozen AHR/rules/checkpoint definitions and bind the new experiment to its sources. Inputs: original notebooks and v4 exports. Outputs: a verified namespace and source identities. Expected runtime: under 15 seconds. No training or generation runs here.

All new execution uses GPU 1. Source notebooks are immutable once the main protocol is frozen. Learned strategic estimates remain UNKNOWN; only declared bounded predicates can be verified. Checkpoints: `checkpoints/<run_id>/`. Resume each stage through its documented notebook with unchanged arguments and sources. Persistent boot disk remains the storage destination; off-instance backup is deferred by the user.

In [ ]:
from pathlib import Path
import copy, datetime as dt, hashlib, json, os, random, sys, time, uuid
import nbformat

V5_ROOT='experiments/ahr_native_v5'
V5_SCHEMA='ahr_native_go_v5'
V5_DEVICE='cuda:1'
V5_NOTEBOOKS=('00_shared.ipynb','01_model_and_search.ipynb','02_verified_curriculum.ipynb','03_learning.ipynb','04_evaluation_and_gates.ipynb','05_budgeted_pipeline.ipynb')

def v5_project(start=None):
    project=Path.cwd().resolve() if start is None else Path(start).resolve()
    while not (project/'notebooks/00_instance_setup_and_recovery.ipynb').is_file():
        if project.parent==project:raise RuntimeError('Run inside AHR Go workspace')
        project=project.parent
    return project

def v5_import_exports(path,namespace):
    book=nbformat.read(Path(path),4);nbformat.validate(book);count=0
    for cell in book.cells:
        if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):
            exec(compile(cell.source,str(path)+':ahr-export','exec'),namespace);count+=1
    if not count:raise RuntimeError('No tagged AHR definitions: '+str(path))
    return count

def v5_load_v4(project,namespace):
    project=Path(project)
    v5_import_exports(project/'experiments/ahr_native_v4/00_shared.ipynb',namespace)
    namespace['v4_load_upstream'](project,namespace)
    for name in ('01_native_ahr_and_tactics.ipynb','02_competitive_training.ipynb','03_gnugo_evaluation.ipynb'):
        v5_import_exports(project/'experiments/ahr_native_v4'/name,namespace)
    namespace['torch'].set_num_threads(1)
    return namespace

def v5_code_sha(path):
    book=nbformat.read(path,4);nbformat.validate(book)
    sources=[c.source for c in book.cells if c.cell_type=='code' and 'ahr-export' in c.metadata.get('tags',[])]
    return hashlib.sha256(json.dumps(sources,ensure_ascii=False).encode()).hexdigest()

def v5_source_identity(project,require_all=True):
    project=Path(project)
    paths=sorted((project/'notebooks').glob('*.ipynb'))
    paths += [project/'experiments/ahr_native_v4'/name for name in ('00_shared.ipynb','01_native_ahr_and_tactics.ipynb','02_competitive_training.ipynb','03_gnugo_evaluation.ipynb')]
    for name in V5_NOTEBOOKS:
        path=project/V5_ROOT/name
        if path.is_file():paths.append(path)
        elif require_all:raise FileNotFoundError(path)
    return {str(p.relative_to(project)):v5_code_sha(p) for p in paths}

v5_sources=v5_source_identity

def v5_digest(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,separators=(',',':'),allow_nan=False).encode()).hexdigest()

def v5_read_json(path):return json.loads(Path(path).read_text())

def v5_report_path(project,name):
    if Path(name).name!=name:raise ValueError('Simple report name required')
    directory=Path(project)/'reports/ahr_native_v5';directory.mkdir(parents=True,exist_ok=True)
    return directory/name

def v5_atomic_json(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temporary=path.with_name('.'+path.name+'.'+uuid.uuid4().hex+'.tmp')
    data=json.dumps(value,sort_keys=True,indent=2,allow_nan=False)+'\n'
    try:
        with temporary.open('w') as stream:
            stream.write(data);stream.flush();os.fsync(stream.fileno())
        os.replace(temporary,path)
        descriptor=os.open(str(path.parent),os.O_DIRECTORY)
        try:os.fsync(descriptor)
        finally:os.close(descriptor)
    finally:
        if temporary.exists():temporary.unlink()
    return path

def v5_seed(seed):
    import numpy as np, torch
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True
    torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False
    random.seed(seed);np.random.seed(seed%(2**32));torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)

def v5_now():return dt.datetime.now(dt.timezone.utc).isoformat()


In [ ]:
PROJECT=v5_project()
v5_load_v4(PROJECT,globals())
assert callable(play) and callable(v4_training_event) and callable(checkpoint_lock)
print({'status':'PASS','workspace':str(PROJECT),'device':V5_DEVICE,'sources_present':len(v5_source_identity(PROJECT,False))})